# Output Parser
**Output Parser**는 대규모 언어 모델(LLM, Large Language Model)의 출력 결과를 **애플리케이션에서 활용할 수 있도록 적절한 형식으로 변환**하는 도구이다.
- LLM은 일반적으로 텍스트 형태로 응답을 생성하지만, 이 텍스트는 그대로 활용하기 어려운 경우가 많다.
- Output Parser는 이러한 **비구조적 텍스트 데이터를 구조화된 데이터로 변환**하여 프로그램에서 활용 가능하도록 만든다.
- 예를 들어, 키워드 리스트를 뽑거나 JSON 형식으로 정보를 변환하는 데 사용된다.

## 주요 Output Parser 종류

1. **CommaSeparatedListOutputParser**
   - 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
   - 예: `"사과, 바나나, 포도"` → `["사과", "바나나", "포도"]`
2. **JsonOutputParser**
   - LLM의 출력이 JSON 형식일 때 이를 Python의 `dict` 객체로 변환한다.
   - JSON(JavaScript Object Notation)은 데이터 구조를 표현하기 위한 경량 포맷이다.
3. **PydanticOutputParser**
   - JSON 데이터를 Python의 [Pydantic](https://docs.pydantic.dev) 모델로 변환한다.
   - Pydantic은 데이터 유효성 검사와 설정 관리에 널리 사용되는 Python 라이브러리이다.
4. **StrOutputParser**
   - 모델의 출력 결과를 단순 문자열로 반환한다.
   - Chat 기반 모델은 Message 객체의 속성으로 LLM 결과를 반환한다. 거기에서 응답 문자열만 추출해서 반환한다.
> `JsonOutputParser`, `PydanticOutputParser` 는 모두 Pydantic을 사용해 데이터 구조(schema)를 정의하고, 해당 구조에 따라 출력을 검증하고 변환한다.

## 주요 메소드
- `parse(text: str)`
  - LLM이 생성한 문자열 응답을 받아 정해진 구조로 변환하여 반환한다.
- `get_format_instructions() -> str`
  - 각 OutputParser가 변환할 수있는 형식으로 LLM이 응답하도록 하는 프롬프트 텍스트를 반환한다.
  - 이 내용을 프롬프트에 넣어서 LLM이 정확한 포맷으로 응답하도록 유도한다.
  
## 참고
- Output Parser는 일반적으로 [`Runnable`](05_chaing_LECL.ipynb#Runnable) 인터페이스를 상속하여 구현되며, `invoke()` 메서드를 통해 실행할 수 있다.
- `invoke()`는 내부적으로 `parse()`를 호출하여 동작한다.
- 필요한 경우 Output Parser를 직접 구현하여 사용자 정의 출력 포맷을 처리할 수도 있다. 

In [ ]:
# ____________을 채워주세요.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 프롬프트 생성
prompt_template = ChatPromptTemplate.from_template(
    "한국의 {topic} 관련된 속담을 {count}개 알려줘."
)
prompt = prompt_template.format(topic="호랑이", count=2)

# LLM 모델 생성
model = ChatOpenAI(model_name="gpt-4o-mini")

# LLM 모델에 prompt를 전달하고 응답 받기.
## prompt -> llm model -> response
response = model.invoke(prompt)

parser = StrOutputParser()  # Message 객체에서 content 속성(메세지)의 값만 추출.
res = parser.invoke(response) # LLM 모델의 응답결과

# prompt_template -> model -> output parser
chain = prompt_template | model | parser

res = chain.invoke({"topic":"사람의 정신력", "count":3})
print(res)

한국에는 사람의 정신력이나 의지를 강조하는 속담이 여러 가지 있습니다. 다음은 그 중 세 가지입니다:

1. **가는 날이 장날** - 하던 일이 잘 풀리거나 뜻밖의 좋은 일이 생겨나는 경우를 나타내는 속담으로, 의지와 노력의 중요성을 시사합니다.

2. **고생 끝에 낙이 온다** - 힘들고 어려운 과정을 겪은 후에 마침내 좋은 결과가 온다는 뜻으로, 인내와 끈기를 강조하는 속담입니다.

3. **하늘은 스스로 돕는 자를 돕는다** - 스스로 노력하고 의지를 가진 사람에게 좋은 결과가 따른다는 의미로, 자기 노력의 중요성을 나타냅니다.

이 속담들은 한국 문화에서 사람의 정신력과 의지를 높이 평가하는 데 사용됩니다.


In [8]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from textwrap import dedent

parser = CommaSeparatedListOutputParser()


prompt_template = ChatPromptTemplate.from_template(
    dedent("""
    # instruction
    {subject}의 이름 다섯개를 나열해주세요.
    
    # output indicator
    {format_instructions}
    """),
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
# partial_variables={변수명:넣을값,..} : 템플릿의 placeholder 변수에 넣을 값을 
#   PromptTemplate객체를 생성할 때 넣겠다. placeholder에 넣을 값이 있는데 함수나 메소드호출을 통해 
#   그 값을 가져와야 하는 경우 사용.

model = ChatOpenAI(model_name="gpt-4o-mini")

format_string = parser.get_format_instructions()
# prompt 생성
prompt = prompt_template.invoke({"subject":"동물"})
# LLM에 요청
response = model.invoke(prompt)

print(response.content)

강아지, 고양이, 토끼, 코끼리, 사자


In [9]:
res = parser.invoke(response)
print(type(res))
print(res)

<class 'list'>
['강아지', '고양이', '토끼', '코끼리', '사자']


In [5]:
# JsonOutputParser
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from textwrap import dedent

class ItemSchema(BaseModel):
    # JSON에 포함될 항목들을 class변수로 정의. 변수명: 타입 = Field(설명)
    name: str = Field(description="제품의 이름")
    info: str = Field(description="제품에 대한 정보")
    release_date: str = Field(description="제품이 출시된 일시. yyyy-mm-dd 형식")
    price: int = Field(description="제품의 한국 가격.")

parser = JsonOutputParser(pydantic_object=ItemSchema)
# print(parser.get_format_instructions())

prompt_template = ChatPromptTemplate.from_template(
    dedent("""
    # instruction
    {name}에 대해서 설명해주세요.
           
    # output indicator
    {format_instructions}
    """),
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

model = ChatOpenAI(model_name="gpt-4o-mini")

p = prompt_template.invoke({"name":"Galaxy S24"})
res = model.invoke(p)
response = parser.invoke(res)

response

{'name': 'Galaxy S24',
 'info': 'Galaxy S24는 삼성전자가 제조한 최신 스마트폰으로, 고급스러운 디자인과 향상된 카메라 성능, 신속한 프로세서, 그리고 사용자 경험을 개선하기 위한 여러 기능을 갖추고 있습니다.',
 'release_date': '2024-02-01',
 'price': 1199000}

In [21]:
# PydanticOutputParser
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=ItemSchema)
# JsonOutputParser와 동일한 format instruction을 생성.
## 응답: JSON ->  Parser: Pydatic Model객체로 변환.
# print(parser.get_format_instructions())

prompt_template = ChatPromptTemplate.from_template(
    dedent("""
    # instruction
    {name}에 대해서 설명해주세요.
           
    # output indicator
    {format_instructions}
    """),
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

model = ChatOpenAI(model_name="gpt-4o-mini")
prompt = prompt_template.invoke({"name":"Mac Book"})
res = model.invoke(prompt)
response = parser.invoke(res)

chain = prompt_template | model | parser

response = chain.invoke({"name":"아이폰"})

type(response)

__main__.ItemSchema

# Chain

**Chain**(체인)은 여러 컴포넌트(요소)를 정해진 순서대로 연결하여 **복잡한 AI 작업을 단계별로 자동화**할 수 있도록 돕는 구조이다.

- 각 컴포넌트는 입력을 받아 특정 처리를 수행한 후 다음 단계로 결과를 전달한다. prompt template -> LLM -> output parser -> 결과 (chain 기법)
- 복잡한 작업을 여러 개의 단순한 단계로 나누고, 각 단계를 순차적으로 실행함으로써 전체 작업을 체계적으로 구성할 수 있다.

## 기본 개념

- 체인은 하나의 LLM 호출에 그치지 않고 **여러 LLM 호출이나 도구 실행을 순차적으로 연결**할 수 있다. -> 순차적 연결이 핵심. (반복, 분기등은 우리가 구현해주어야 함.)
- 예를 들어, 사용자의 질문 → 검색 → 요약 → 응답 생성 같은 일련의 작업을 체인으로 구성할 수 있다.
- 체인을 사용하면 코드의 재사용성과 유지 보수성이 향상된다.

## LangChain에서의 Chain 구성 방식

LangChain은 다음 두 가지 방식을 통해 체인을 구성할 수 있다.

### 1. Off-the-shelf Chains 방식 (클래식 방식)

- LangChain에서 제공하는 **미리 정의된 Chain 클래스**(예: `LLMChain`, `SequentialChain`, `SimpleSequentialChain`)를 활용하는 방식이다.
- 이 방식은 LangChain의 **초기 구조**이며, 대부분의 클래스는 현재 **더 이상 사용되지 않음(deprecated)** 상태이다.

> 현재 LangChain에서는 이 방식을 권장하지 않는다.

### 2. LCEL (LangChain Expression Language) 방식

- 체인을 함수형 방식으로 선언할 수 있는 **표현식 기반의 체인 구성 언어**이다.
- LCEL 방식은 간결하고 선언적인 문법을 제공하여 **직관적이고 확장성 있는 체인 구성**이 가능하다.
- `Runnable`이라는 공통 인터페이스를 기반으로 다양한 요소를 조합하여 체인을 구성한다.
- 체인의 각 구성 요소는 `invoke()` 메서드로 실행된다.

In [10]:
# off-the-shelf 방식
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain import LLMChain

load_dotenv()

prompt_template = PromptTemplate(
    template="{item}에 어울리는 이름 {count}개를 만들어 주세요."
)
model = ChatOpenAI(model_name="gpt-4o-mini")
parser = StrOutputParser()

chain = LLMChain(
    prompt=prompt_template,
    llm=model, 
    output_parser=parser
)
response = chain.invoke({"item":"가방", "count":5})

response

C:\Users\Playdata\AppData\Local\Temp\ipykernel_17392\142610202.py:16: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


{'item': '가방',
 'count': 5,
 'text': '물론입니다! 가방에 어울리는 이름 다섯 개를 제안해 드릴게요.\n\n1. **스타일리시 에코 (Stylish Eco)** - 친환경 소재로 만들어진 멋진 가방.\n2. **모던 캐리 (Modern Carry)** - 현대적인 디자인의 다용도 가방.\n3. **소프트 프레임 (Soft Frame)** - 부드러운 재질의 편안한 가방.\n4. **클래식 터치 (Classic Touch)** - 클래식한 느낌의 우아한 가방.\n5. **어반 트레일러 (Urban Trailer)** - 도시적인 감성을 담은 실용적인 가방.\n\n이름들이 도움이 되길 바랍니다!'}

# [LCEL](https://python.langchain.com/docs/how_to/#langchain-expression-language-lcel) (LangChain Expression Language)
- LCEL은 LangChain의 핵심 기능인 체인(Chain)을 더욱 간결하고 유연하게 구성할 수 있도록 고안된 **선언형 체인(chain) 구성 언어**이다.
- 파이프 연산자 `|`를 사용해 선언적 방법으로 여러 작업을 연결한다.
- 체인을 구성하는 각 요소는 `Runnable` 타입이나 `함수등 Callable객체`으로, 체인 내에서 실행 가능한 단위이다.
- 각 단계는 invoke() 메서드를 통해 실행되며, 앞 단계의 출력이 다음 단계의 입력으로 자동 전달된다.
    - [Runnable 컴포넌트별 입출력 타입](https://python.langchain.com/docs/concepts/runnables/#input-and-output-types)
    - 각 컴포넌트의 input과 output 타입에 맞춰 값이 전달되도록 한다.
- https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel

## [Runnable](https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.base.Runnable.html)
- LangChain의 Runnable은 실행 가능한 작업 단위를 캡슐화한 개념으로, 데이터 흐름의 각 단계를 정의하고 **체인(chain) 에 포함 되어**  복잡한 작업의 각 단계를 수행 한다.
- Chain을 구성하는 class들은 Runnable의 상속 받아 구현한다.
- **Prompt Template클래스**, **Chat 모델, LLM 모델 클래스**, **Output Parser 클래스** 등 다양한 컴포넌트가 Runnable을 상속받아 구현된다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 호출, 출력 파싱 등)을 수행하는 독립적인 컴포넌트이다.
    - 각 컴포넌트는 독립적으로 테스트 및 재사용이 가능하며, 조합하여 복잡한 체인을 구성할 수 있다.
- 체인 연결 및 작업 흐름 관리:
    - Runnable은 체인(chain, 일련의 연결된 작업 흐름)을 구성하는 기본 단위로 사용된다.
    - LangChain Expression Language(LCEL)를 사용하면 | 연산자를 통해 여러 Runnable을 쉽게 연결할 수 있다.
    - 입력과 출력의 형식을 일관되게 유지하여 각 단계가 자연스럽게 연결된다.
- 모듈화 및 디버깅 용이성:
    - 각 단계가 명확히 분리되어 문제 발생 시 어느 단계에서 오류가 발생했는지 쉽게 확인할 수 있다.
    - 복잡한 작업을 작은 단위로 나누어 체계적으로 관리할 수 있다.
      
### Runnable의 표준 메소드
- 모든 Runnable이 구현하는 공통 메소드
    - `invoke()`: 단일 입력을 처리하여 결과를 반환.
    - `batch()`: 여러 입력 데이터들을 한 번에 처리.
    - `stream()`: 입력에 대해 스트리밍 방식으로 응답을 반환.
    - `ainvoke()`: 비동기 방식으로 입력을 처리하여 결과를 반환.

### Runnable의 주요 구현체(하위 클래스)

- `RunnableSequence`
    - 여러 `Runnable`을 순차적으로 연결하여 실행하는 구성이다.
    - 각 단계의 출력이 다음 단계의 입력으로 전달된다.
    - LCEL을 사용하여 체인을 구성할 경우 자동으로 `RunnableSequence`로 변환된다.
-  `RunnablePassthrough`
    - 입력 데이터를 가공하지 않고 그대로 다음 단계로 전달하는 `Runnable`이다.
    - 선택적으로 미리 정의된 key-value 쌍을 함께 전달할 수 있다.

- `RunnableParallel`
    - 여러 `Runnable`을 병렬로 실행한 후, 결과를 결합하여 다음 단계로 전달한다.
    - 병렬 처리를 통해 처리 속도를 개선할 수 있다.

- `RunnableLambda`
    - 일반 함수 또는 `lambda` 함수를 `Runnable`로 변환하여 체인에 포함할 수 있다.
    - 사용자 정의 함수로 동작을 확장할 때 유용하다.

#### Runnable 예제

In [11]:
# 기본 체인 구성: prompt_template -> model -> output parser
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

# role: system, user/human, ai/assistant
#       system: 채팅  전체에 적용되는 공통 지침을 지정하는 role
prompt_template = ChatPromptTemplate(
    messages=[
        ("system", "당신은 오랜 경력의 한국 관광 가이드입니다. 여행객들에게 설명하듯이 친절하게 답변을 해주세요."),
        ("human", "{query}")
    ]
)
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=1.0)

guide_chain = prompt_template | model | StrOutputParser()

print(type(guide_chain)) # RunnableSequence: Runnable 타입 
#                           ==> chain도 다른 chain의 구성요소로 포함될 수있다.

query = "서울에서 꼭 가봐야되는 여행지를 세 곳만 알려줘."
response = guide_chain.invoke({"query":query})

print(response)

<class 'langchain_core.runnables.base.RunnableSequence'>
안녕하세요! 서울에 오신 것을 환영합니다. 서울은 역사와 현대가 조화롭게 어우러진 멋진 도시입니다. 꼭 가봐야 할 여행지를 세 곳 추천해드리겠습니다.

1. **경복궁**: 서울의 대표적인 고궁 중 하나인 경복궁은 조선 왕조의 대표적인 궁궐입니다. 아름다운 건축물과 탁 트인 정원이 매력적이며, 특히 매일 열리는 수문장 교대식은 꼭 보셔야 해요. 경복궁 근처에 있는 국립민속박물관과 국립고궁박물관도 함께 방문하시면 좋습니다.

2. **남산서울타워 (N서울타워)**: 서울의 상징적인 랜드마크인 남산서울타워는 전망대에서 서울 시내의 전경을 한눈에 내려다볼 수 있는 훌륭한 장소입니다. 저녁에 가면 아름다운 야경을 감상할 수 있고, 탑 주변의 산책길과 사랑의 자물쇠도 놓치지 마세요.

3. **홍대 거리**: 젊음과 문화가 가득한 홍대는 예술과 음악의 중심지로, 다양한 카페, 갤러리, 거리공연을 즐길 수 있습니다. 특히 주말에는 꿈의 거리로 변신해 다채로운 플리마켓과 길거리 공연이 열리니, 에너지가 넘치는 분위기를 느껴보세요.

이 외에도 서울에는 가볼 만한 곳이 정말 많답니다! 즐거운 여행 되세요!


In [ ]:
while True:
    query = input("질문:")
    if query == "!quit":
        break
    resp = guide_chain.invoke({"query":query})
    print("User:", query)
    print("AI:", resp)
    print("-"* 50)

#### RunnableLambda 예제

In [12]:
from langchain_core.runnables import RunnableLambda
# RunnableLambda(함수) -> 함수를 실행하는 Runnable을 생성.
my_runnable2 = RunnableLambda(lambda input_data : f"{input_data}를 한 문장으로 설명해줘.")
my_runnable2.invoke("LLM") 

chain = my_runnable2 | model
chain.invoke("LLM")

AIMessage(content='LLM(대형 언어 모델)은 방대한 양의 텍스트 데이터를 기반으로 언어 이해와 생성 능력을 갖춘 인공지능 모델입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 18, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62a23a81ef', 'id': 'chatcmpl-BjjWtx2AgxHuOPP7J0nzx0gOYA4uM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--4e4f8790-6910-41d7-988e-9beba2a25e4f-0', usage_metadata={'input_tokens': 18, 'output_tokens': 34, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

#### RunnablePassthrough 예제

In [17]:
# 1 앞 Runnable이 처리한 결과를 다음 Runnable에 그대로 전달.
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke("안녕하세요")
type(RunnablePassthrough().invoke({"Key":"value"}))
# type(RunnablePassthrough().invoke([1, 2, 3]))

dict

In [18]:
# 2. 앞 Runnable이 처리한 결과에 Item을 추가해서 다음 Runnable에 전달.
#   -> 입력으로 dictionary 받아서 거기에 item을 추가.
# RunnablePassthrough.assign(key1=Runnable, key2=Runnable, ...)
#  - 받은 dictionary에 "key1":Runnable반환값, "key2":Runnable반환값, .. 추가해서 다음으로 전달.
address_runnable = RunnableLambda(lambda x: "서울시 금천구")  #"서울시 금천구" 를 반환.
phone_runnable = RunnableLambda(lambda x: "010-1111-2222")

RunnablePassthrough.assign(address=address_runnable, phone=phone_runnable).invoke({"name":"홍길동"})

{'name': '홍길동', 'address': '서울시 금천구', 'phone': '010-1111-2222'}

#### RunnableSequence 예제

In [ ]:
from langchain_core.runnables import RunnableSequence

run1 = RunnableLambda(lambda x: x + 1)
run2 = RunnableLambda(lambda x: x * 2)

chain = run1 | run2
print(type(chain))
chain.invoke(30)

chain2 = RunnableSequence(run1, run2)  #(prompt_template, model, output_parser)
chain2.invoke(100)

<class 'langchain_core.runnables.base.RunnableSequence'>


202

#### RunnableParallel 예제

In [20]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

run1 = RunnableLambda(lambda x: x + 1)
run2 = RunnableLambda(lambda x: x * 2)
run3 = RunnableLambda(lambda x: x // 3)

runnable = RunnableParallel(
    {
        "result1":run1,
        "result2":run2,
        "result3":run3,
        "result4":RunnablePassthrough()  # 앞에서 받은 값을 그대로 다음에 전달.
    }
)
# Runnable들을 각각 실행하고 그 결과를 key에 할당한 Dictionary를 반환.
runnable.invoke(20)

{'result1': 21, 'result2': 40, 'result3': 6, 'result4': 20}

#### LCEL Chain 예제

In [ ]:
# 아래 구성에 맞게 code를 만들어 주세요.

In [ ]:
# 음식 이름을 받아서 레시피를  출력하는 chain을 구성
# prompt template -> model -> output parser 
# response = food_chain.invoke({"food":"pasta"})

In [ ]:
# 번역 chain -> 입력된 내용을 지정한(입력한) 언어로 번역하는 체인.
# prompt template -> model -> output parser 
# 안녕하세요->독일어로 번역한 응답.
